# LongFlow — Gate Night 2 (teacher-only: N8 rate sweep + text-window probe)

Runtime: **L4 GPU**. Teacher-only (no flow head, no training). ~90–120 min, ~$2–3.
Follows finding **N8** (`docs/negative-results.md`): 1.5× rate inflation on a
3229-word script. Pre-registered criteria: `experiments/p1_flow_head/NOTES.md`
(Gate Night 2 entry).

| cell | check | decides |
|---|---|---|
| 2 | build nested prefix scripts (100/500/1500/full words, shared first ~100 words) | controlled dose-response design |
| 4 | rate sweep: 2 voice prompts × 4 script lengths, seed=0 | **is upcoming-text length the pacing driver?** (H1) |
| 6 | chunked text reveal: same full script in ~320-word chunks, same prompt each chunk | **does capping visible text restore natural rate, and what does it cost in identity at the seams?** (H2) |

Colab generates; Mac scores (`score_gate_night2.py`). Download
`gate_night2_bundle.zip` at the end. Every generation is seeded (teacher
determinism proven, Gate Night 1 cell 3) — the `full` condition is a seeded
replication of the N8 endurance run. Captures saved to Drive
(`longflow_longctx_probe/`) double as E4 context-length-OOD conditions.


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, sys, time
import numpy as np
import soundfile as sf

if not os.path.exists("/content/LongFlow/src"):
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone LongFlow or drag longflow_bundle.zip + unzip"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.cache.capture import SampleCapture, save_utterance

TRAIN_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_cache"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
PROBE_DIR = "/content/drive/MyDrive/longflow_longctx_probe"
os.makedirs(PROBE_DIR, exist_ok=True)
OUT = "/content/gate_night2"
os.makedirs(OUT, exist_ok=True)

def gen_inputs(text, prompt_wav):
    inputs = processor(text=[f"Speaker 1: {text}\n"], voice_samples=[[prompt_wav]],
                       return_tensors="pt", padding=True)
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

def generate_seeded(text, prompt_wav, max_new, seed=0, capture_path=None, utt_id=None):
    torch.manual_seed(seed)
    t0 = time.time()
    if capture_path:
        with SampleCapture(model) as cap, torch.inference_mode():
            out = model.generate(**gen_inputs(text, prompt_wav), tokenizer=processor.tokenizer,
                                 cfg_scale=1.3, max_new_tokens=max_new)
        save_utterance(cap.to_utterance(utt_id, text, meta={"kind": "rate_sweep"}), capture_path)
        frames = cap.num_frames
    else:
        with torch.inference_mode():
            out = model.generate(**gen_inputs(text, prompt_wav), tokenizer=processor.tokenizer,
                                 cfg_scale=1.3, max_new_tokens=max_new)
        frames = None
    wav = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
    return wav, time.time() - t0, frames

report = {"sweep": [], "chunked": []}
print("READY")


In [ ]:
# Same sentence pool + order as the N8 endurance run (Gate Night 1 cell 7),
# so `full` is a seeded replication and every condition is a prefix of it.
sents = []
for f in sorted(glob.glob(f"{TRAIN_CACHE_DIR}/*.pt"))[-300:]:
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
pool = sents[:200]

def prefix_script(target_words):
    words, take = 0, []
    for s in pool:
        take.append(s); words += len(s.split())
        if words >= target_words: break
    return " ".join(take)

SCRIPTS = {100: prefix_script(100), 500: prefix_script(500),
           1500: prefix_script(1500), "full": " ".join(pool)}
for k, v in SCRIPTS.items():
    print(k, len(v.split()), "words")
# Nested prefixes: all four conditions open with the IDENTICAL first ~100 words.
# The Mac scorer times that shared segment in each render — same text, same
# voice, only the amount of UPCOMING script differs. That isolates H1.
report["script_words"] = {str(k): len(v.split()) for k, v in SCRIPTS.items()}
report["shared_prefix_words"] = len(SCRIPTS[100].split())


## 3. Rate sweep — is upcoming-text length the pacing driver? (H1)

N8's inflation is present in the first 30 s, before any audio accumulates —
so the suspect is the *length of the visible script*, not rollout history.
Render the identical opening ~100 words under 4 script lengths × 2 voice
prompts, seed=0. If the same words take materially less time when more text
follows them, H1 is confirmed. ~60–90 min, the bulk of the night.


In [ ]:
MAXTOK = {100: 800, 500: 2500, 1500: 6000, "full": 12000}
prompts = sorted(glob.glob(f"{EVAL_CACHE_DIR}/*_prompt.wav"))[:2]
assert prompts, "no *_prompt.wav in eval cache"
report["prompts"] = [os.path.basename(p) for p in prompts]
for pi, prompt in enumerate(prompts):
    for key, script in SCRIPTS.items():
        tag = f"p{pi}_{key}"
        wav, wall, frames = generate_seeded(
            script, prompt, MAXTOK[key], seed=0,
            capture_path=f"{PROBE_DIR}/rate_sweep_{tag}.pt", utt_id=f"rate_sweep_{tag}")
        sf.write(f"{OUT}/sweep_{tag}.wav", wav, 24000)
        row = {"tag": tag, "prompt": os.path.basename(prompt), "condition": str(key),
               "script_words": len(script.split()), "audio_s": round(len(wav)/24000, 1),
               "wall_s": round(wall), "frames": frames,
               "near_cap": frames is not None and frames >= MAXTOK[key] - 16}
        report["sweep"].append(row)
        print(row, flush=True)


## 5. Chunked text reveal — does capping visible text restore natural rate? (H2)

Same full script, but the model only ever *sees* ~320 words at a time: one
`generate()` per chunk, **same original voice prompt every chunk** (identity
held constant so the text-window effect is isolated; seam continuity is the
measured cost, not a confound). If chunked wpm lands near the standalone
500-word condition and well under the full-script rate, text windowing cures
the pacing defect with zero training — and the seam-ECAPA number tells us
whether the naive version is listenable or needs the KV-cache implementation.


In [ ]:
chunks, cur, w = [], [], 0
for s in pool:
    cur.append(s); w += len(s.split())
    if w >= 320:
        chunks.append(" ".join(cur)); cur, w = [], 0
if cur:
    chunks.append(" ".join(cur))
print(len(chunks), "chunks:", [len(c.split()) for c in chunks])

prompt = prompts[0]
pieces = []
for ci, chunk in enumerate(chunks):
    wav, wall, _ = generate_seeded(chunk, prompt, 1500, seed=0)
    sf.write(f"{OUT}/chunk_{ci:02d}.wav", wav, 24000)
    pieces.append(wav)
    report["chunked"].append({"chunk": ci, "words": len(chunk.split()),
                              "audio_s": round(len(wav)/24000, 1), "wall_s": round(wall)})
    print(ci, len(chunk.split()), "w ->", round(len(wav)/24000, 1), "s", flush=True)
sf.write(f"{OUT}/chunked_full.wav", np.concatenate(pieces), 24000)
print("chunked_full:", round(sum(len(p) for p in pieces)/24000/60, 2), "min")


## 6. Bundle

Grab `gate_night2_bundle.zip`. Mac side:
`python experiments/p1_flow_head/score_gate_night2.py` after extracting to
`experiments/p1_flow_head/audio/gate_night2/`.


In [ ]:
import zipfile
json.dump(report, open(f"{OUT}/gate_night2_report.json", "w"), indent=2)
with zipfile.ZipFile("/content/gate_night2_bundle.zip", "w") as z:
    for f in glob.glob(f"{OUT}/*"):
        z.write(f, os.path.basename(f))
print("download /content/gate_night2_bundle.zip")
